In [ ]:
import polars as pl
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from optbinning import BinningProcess
import lightgbm as lgb
import mlflow
import warnings
warnings.filterwarnings('ignore')

mlflow.set_tracking_uri('mlruns')
mlflow.set_experiment('task4')
RUN_TAG = 'mar23'

In [ ]:
it = pl.read_csv('task4/data/IT.csv')
oot = pl.read_csv('task4/data/OOT.csv')

var_cols = [c for c in it.columns if c.startswith('VAR_')]
cat_cols = [c for c in var_cols if it[c].dtype == pl.String]
num_cols = [c for c in var_cols if c not in cat_cols]
print(f'Vars: {len(var_cols)}, Num: {len(num_cols)}, Cat: {len(cat_cols)}')
print(f'Cat cols: {cat_cols}')

it = it.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))
oot = oot.with_columns(pl.col('TIME').str.to_datetime().alias('TIME_DT'))

cutoff = it.sort('TIME_DT')['TIME_DT'].quantile(0.8)
train = it.filter(pl.col('TIME_DT') <= cutoff)
val = it.filter(pl.col('TIME_DT') > cutoff)
print(f'Train: {train.shape[0]}, Val: {val.shape[0]}')

In [ ]:
from sklearn.preprocessing import LabelEncoder

X_train = train.select(var_cols).to_pandas()
y_train = train['TARGET'].to_numpy()
X_val = val.select(var_cols).to_pandas()
y_val = val['TARGET'].to_numpy()
X_oot = oot.select(var_cols).to_pandas()

encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    all_vals = list(X_train[c].dropna().unique()) + list(X_val[c].dropna().unique()) + list(X_oot[c].dropna().unique())
    le.fit(list(set(all_vals)) + ['__missing__'])
    for df in [X_train, X_val, X_oot]:
        df[c] = df[c].fillna('__missing__')
        df[c] = le.transform(df[c])
    encoders[c] = le

print(f'Encoded {len(cat_cols)} categorical columns')

In [ ]:
bp = BinningProcess(
    variable_names=var_cols,
    categorical_variables=cat_cols,
    min_prebin_size=0.02,
    min_bin_size=0.05,
    max_n_bins=10,
    selection_criteria={'iv': {'min': 0.02}}
)
bp.fit(X_train.values, y_train)
selected = list(bp.get_support(names=True))
print(f'Selected (IV>=0.02): {len(selected)}')

summary = bp.summary().sort_values('iv', ascending=False)
print(summary[['name', 'iv', 'dtype']].to_string())

In [ ]:
X_tr_woe = bp.transform(X_train.values, metric='woe')
X_va_woe = bp.transform(X_val.values, metric='woe')
X_oo_woe = bp.transform(X_oot.values, metric='woe')

best_gini_lr = 0
best_lr = None
best_C = None

for C in [0.01, 0.1, 0.5, 1.0]:
    lr = LogisticRegression(C=C, max_iter=1000, solver='lbfgs')
    lr.fit(X_tr_woe, y_train)
    p = lr.predict_proba(X_va_woe)[:, 1]
    g = 2 * roc_auc_score(y_val, p) - 1
    print(f'WOE+LR C={C}: val_gini={g:.4f}')
    if g > best_gini_lr:
        best_gini_lr, best_lr, best_C = g, lr, C

p_lr_val = best_lr.predict_proba(X_va_woe)[:, 1]
print(f'Best LR: C={best_C}, val_gini={best_gini_lr:.4f}')

In [ ]:
lgb_tr = lgb.Dataset(X_train, y_train, free_raw_data=False, categorical_feature=cat_cols)
lgb_va = lgb.Dataset(X_val, y_val, reference=lgb_tr, free_raw_data=False)

params = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
    'feature_pre_filter': False,
    'num_leaves': 31, 'learning_rate': 0.05,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
    'min_child_samples': 100,
}

model = lgb.train(params, lgb_tr, num_boost_round=1000, valid_sets=[lgb_va],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

p_lgb_val = model.predict(X_val)
val_gini_lgb = 2 * roc_auc_score(y_val, p_lgb_val) - 1
print(f'LGB raw: val_gini={val_gini_lgb:.4f}')

In [ ]:
best_ens_g = 0
best_w = 0
for w in np.arange(0.0, 1.05, 0.05):
    p_ens = w * p_lr_val + (1 - w) * p_lgb_val
    g = 2 * roc_auc_score(y_val, p_ens) - 1
    if g > best_ens_g:
        best_ens_g, best_w = g, w

print(f'Best ensemble: w_lr={best_w:.2f}, val_gini={best_ens_g:.4f}')

results = {'WOE_LR': best_gini_lr, 'LGB_raw': val_gini_lgb, 'Ensemble': best_ens_g}
best_name = max(results, key=results.get)
val_gini = results[best_name]
print(f'Best: {best_name} = {val_gini:.4f}')
for k, v in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {k}: {v:.4f}')

In [ ]:
if best_name == 'WOE_LR':
    p_oot_final = best_lr.predict_proba(X_oo_woe)[:, 1]
elif best_name == 'LGB_raw':
    p_oot_final = model.predict(X_oot)
else:
    p_oot_final = best_w * best_lr.predict_proba(X_oo_woe)[:, 1] + (1 - best_w) * model.predict(X_oot)

preds = pl.DataFrame({'ID_APPLICATION': oot['ID_APPLICATION'], 'SCORE': p_oot_final})
preds.write_csv('task4/predictions.csv')
print(f'OOT predictions: {preds.shape}')

with mlflow.start_run(run_name='v1_baseline'):
    mlflow.log_param('model', best_name)
    mlflow.log_param('n_features', len(var_cols))
    mlflow.log_param('n_selected_woe', len(selected))
    mlflow.log_metric('val_gini', val_gini)
    mlflow.log_metric('val_gini_lr', best_gini_lr)
    mlflow.log_metric('val_gini_lgb', val_gini_lgb)
    mlflow.log_metric('val_gini_ens', best_ens_g)
    mlflow.set_tag('task', 'task4')
    mlflow.set_tag('run_tag', RUN_TAG)
print(f'val_gini: {val_gini:.4f}')